In [1]:
import os
import json 
import getpass
import logging
from pathlib import Path
from time import sleep
import subprocess
from datetime import datetime
import sys
import re

from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_community.callbacks import get_openai_callback
from langchain.tools import BaseTool

from cluster_tester import ClusterTester
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from itertools import combinations
import traceback

from typing import TypedDict, Dict, List, Any

from PyDI.io import load_xml, load_parquet, load_csv
from PyDI.utils import DataProfiler
from blocking_tester import BlockingTester
from matching_tester import MatchingTester
from PyDI.entitymatching import RuleBasedMatcher
from schema_matching_node import run_schema_matching

from dotenv import load_dotenv
load_dotenv()

/Users/abd/Developer/team_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# print(sys.getrecursionlimit())

In [3]:
# sys.setrecursionlimit(5000)
# print(sys.getrecursionlimit())

## Initialize

In [4]:
OUTPUT_DIR = "output/"
INPUT_DIR = "input/"

INCLUDE_DOCS = False # IMPORTANT: Use carefully since token usage is increased drastically with documentation
USE_LLM = "gpt"
#USE_LLM = "gemini"
#USE_LLM = "groq"

In [5]:
if USE_LLM == "gemini": # or USE_LLM == "gemini_broken":
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
elif USE_LLM == "groq":
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
elif USE_LLM == "gpt":
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [6]:
logging.basicConfig(filename= OUTPUT_DIR + 'agent.log',
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%H:%M:%S',
                    level=logging.DEBUG,
                    encoding='utf-8')

## Utilities

In [7]:
def load_dataset(path):
    # check file exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at: {path}")
    ext = os.path.splitext(path)[1].lower()

    # load dataset according to extension
    if ext == ".parquet":
        df = load_parquet(path)
    elif ext == ".csv":
        df = load_csv(path)
    elif ext == ".xml":
        df = load_xml(path, nested_handling="aggregate")
    else:
        raise ValueError(f"Unsupported format: {ext}. Supported: .csv, .parquet, .xml")
    return df
    

## Tools

In [8]:
class ProfileDatasetTool(BaseTool):
    name: str = "profile_dataset"
    description: str = """
        A tool that takes the path as a argument called `path` of type str of the dataset file as string and performs data analysis. 
        A JSON string is returned with the profile data.
    """
    
    def _run(self, path) -> str:   
        return self.create_profile(path)

    def create_profile(self, path):

        if not path or not isinstance(path, str):
                raise ValueError("ProfileDatasetTool requires a single path string argument.")

        df = load_dataset(path)    
        
        if df is None or getattr(df, "empty", False):
            # return a structured JSON error string (LLM will see this as content)
            return json.dumps({"error": f"Dataset at {path} loaded as empty or failed to load."})

        profiler = DataProfiler()
        profile = profiler.summary(df, print_summary=False)

        # ensure to return a JSON string (your docstring promised JSON string)
        try:
            profile_json = json.dumps(profile, default=str)
        except Exception:
            # fallback: convert to str if not json-serializable
            profile_json = json.dumps({"profile_str": str(profile)})

        return profile_json
        

In [9]:
class SearchDocumentationTool(BaseTool):
    """
    A tool to search the PyDI documentation vector database.
    """
    name: str = "search_documentation"
    description: str = (
        "Searches the PyDI library documentation for a given query. "
        "Use this to find information about functions, classes, or how-to instructions. "
        "The input should be a specific question about the library."
    )
    call_count: int = 0

    def _clean_text(self, text: str) -> str:
        """
        Cleans and normalizes LLM or tool output text.
        Safe for LangChain AIMessage.content or ToolMessage.content.
        """

        if text is None:
            return ""

        text = str(text)
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"\n\s*\n+", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = text.strip()

        return text


    def _run(self, query: str) -> str:
        """Executes the documentation search."""
        self.call_count += 1
        print(f"[SEARCH TOOL CALL {self.call_count}]: Asking '{query}'")
        
        embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
        vector_db_path = os.path.join(INPUT_DIR, "api_documentation/pydi_apidocs_vector_db/")

        if not os.path.exists(vector_db_path):
            error_msg = f"Error: Vector DB not found at {vector_db_path}"
            # self._save_query_response(query, error_msg)
            return error_msg
        try:
            db = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)
            docs = db.similarity_search(query, k=4)
            
            if not docs:
                response = "No relevant documentation found for your query."
            else:
                response = "\n\n-----\n\n".join([r.page_content for r in docs])
                response = self._clean_text(response)
            
            # Save query and response to JSON file
            # self._save_query_response(query, response)
            
            return response
        except Exception as e:
            error_response = f"An error occurred while searching documentation: {str(e)}\n{traceback.format_exc()}"
            # self._save_query_response(query, error_response)
            return error_response

    def reset(self):
        """Resets the call count for a new agent run."""
        self.call_count = 0

## Agents

In [10]:
class SimpleModelAgentState(TypedDict):
    datasets: list
    entity_matching_testsets: list
    fusion_testset: str
    blocking_config: Dict  # NEW: Blocking config from BlockingTester
    matching_config: Dict  # NEW: Matching config from MatchingTester
    matcher_mode: str  # NEW: Matcher mode (rule_based/ml/auto)
    schema_correspondences: Dict  # NEW: Schema matching results
    
    data_profiles: Dict

    messages: List[AnyMessage] # Save conversation history during research
    eval_messages: List[AnyMessage] # Save conversation history during evaluation

    cluster_analysis_result: Dict
    cluster_recommendation_applied: str | None
    cluster_analysis_history: list
    matching_config_updated: bool
    
    integration_pipeline_code: str
    pipeline_execution_result: str
    pipeline_execution_attempts: int

    evaluation_code: str
    evaluation_execution_result: str
    evaluation_execution_attempts: int

    evaluation_attempts: int
    evaluation_metrics: Dict
    evaluation_analysis: str
    evaluation_reasoning_history: list
    execution_trace: list
    node_counts: Dict
    node_token_usage: Dict

class SimpleModelAgent:
    
    def __init__(self, model, tools: Dict[str, BaseTool]):
        # initialize logger
        self.logger = logging.getLogger()
        self.logger.setLevel(logging.INFO)
        log_path = os.path.join(OUTPUT_DIR, "agent_run.log")
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        if not any(isinstance(h, logging.FileHandler) and h.baseFilename == os.path.abspath(log_path) for h in self.logger.handlers):
            file_handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
            formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
            file_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)
        
        self.token_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "total_cost": 0.0}
        self.node_token_usage = {}
        self.node_token_usage_totals = {}
        self.log_path = log_path
        self.tools = tools
        self.model = model.bind_tools(list(self.tools.values()))
        self.base_model = model

        # prepare the StateGraph
        graph = StateGraph(SimpleModelAgentState)

        # create nodes
        graph.add_node("match_schemas", self.match_schemas)
        graph.add_node("profile_data", self.profile_data)
        graph.add_node("run_blocking_tester", self.run_blocking_tester)
        graph.add_node("run_matching_tester", self.run_matching_tester)
        graph.add_node("pipeline_adaption", self.pipeline_adaption)
        graph.add_node("execute_pipeline", self.execute_pipeline)
        graph.add_node("cluster_analysis", self.cluster_analysis)
        graph.add_node("evaluation_adaption", self.evaluation_adaption)
        graph.add_node("execute_evaluation", self.execute_evaluation)
        graph.add_node("evaluation_decision", self.evaluation_decision)
        graph.add_node("evaluation_reasoning", self.evaluation_reasoning)
        graph.add_node("save_results", self.save_results)

        # create edges
        graph.add_edge("match_schemas", "profile_data")
        graph.add_edge("profile_data", "run_blocking_tester")
        graph.add_edge("run_blocking_tester", "run_matching_tester")
        graph.add_edge("run_matching_tester", "pipeline_adaption")
        
        graph.add_conditional_edges(
            "pipeline_adaption",
            self.should_continue_research,
            {
                "continue": "pipeline_adaption",
                "end": "execute_pipeline"
            }
        )
        
        graph.add_conditional_edges(
            "execute_pipeline",
            lambda state: (
                "cluster_analysis"
                if state.get("pipeline_execution_result", "")
                .lower()
                .startswith("success")
                else (
                    "pipeline_adaption"
                    if state.get("pipeline_execution_attempts", 0) < 3
                    else END
                )
            ),
            {
                "cluster_analysis": "cluster_analysis",
                "pipeline_adaption": "pipeline_adaption",
                END: END,
            },
        )

        # Edge from cluster analysis to evaluation or pipeline adaption (if recommendation pending)
        graph.add_conditional_edges(
            "cluster_analysis",
            self._cluster_analysis_next_step,
            {
                "pipeline_adaption": "pipeline_adaption",
                "evaluation_adaption": "evaluation_adaption",
            },
        )

        graph.add_edge("evaluation_adaption", "execute_evaluation")

        graph.add_conditional_edges(
            "execute_evaluation",
            lambda state: (
                "evaluation_decision" 
                if isinstance(state.get("evaluation_execution_result", ""), str)
                   and state["evaluation_execution_result"].lower().startswith("success")
                else "evaluation_adaption"
                if state.get("evaluation_attempts", 0) < 3
                else END
            ),
            {
                "evaluation_adaption": "evaluation_adaption",
                "evaluation_decision": "evaluation_decision",
                END: END
            }
        )


        graph.add_conditional_edges(
            "evaluation_decision",
            lambda state: (
                "evaluation_reasoning"
                if state.get("evaluation_metrics", {}).get("overall_accuracy", 0) < 0.85
                   and state.get("evaluation_attempts", 0) < 3
                else "save_results"
            ),
            {
                "evaluation_reasoning": "evaluation_reasoning",
                "save_results": "save_results",
                END: END
            }
        )

        graph.add_edge("evaluation_reasoning", "pipeline_adaption")
        graph.add_edge("save_results", END)

        graph.set_entry_point("match_schemas")
        self.graph = graph.compile()

    def _invoke_with_usage(self, message, tag):
        with get_openai_callback() as cb:
            result = self.model.invoke(message)
        self.token_usage["prompt_tokens"] += cb.prompt_tokens
        self.token_usage["completion_tokens"] += cb.completion_tokens
        self.token_usage["total_tokens"] += cb.total_tokens
        self.token_usage["total_cost"] += cb.total_cost
        usage = {"prompt_tokens": cb.prompt_tokens, "completion_tokens": cb.completion_tokens, "total_tokens": cb.total_tokens, "total_cost": cb.total_cost}
        self.node_token_usage.setdefault(tag, []).append(usage)
        totals = self.node_token_usage_totals.setdefault(tag, {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "total_cost": 0.0})
        totals["prompt_tokens"] += cb.prompt_tokens
        totals["completion_tokens"] += cb.completion_tokens
        totals["total_tokens"] += cb.total_tokens
        totals["total_cost"] += cb.total_cost
        print(f"TOKEN USAGE ({tag}):")
        print(f"   Prompt tokens: {cb.prompt_tokens:,}")
        print(f"   Completion tokens: {cb.completion_tokens:,}")
        print(f"   Total tokens: {cb.total_tokens:,}")
        print(f"   Estimated cost: ${cb.total_cost:.6f}")
        return result

    def _print_total_usage(self):
        t = self.token_usage
        print("TOTAL TOKEN USAGE:")
        print(f"   Prompt tokens: {t['prompt_tokens']:,}")
        print(f"   Completion tokens: {t['completion_tokens']:,}")
        print(f"   Total tokens: {t['total_tokens']:,}")
        print(f"   Estimated cost: ${t['total_cost']:.6f}")


    def _apply_matching_weights(self, pipeline_code: str, matching_config: Dict[str, Any]) -> str:
        """Apply matching_config weights to generated pipeline code."""
        import re

        if not matching_config:
            return pipeline_code
        strategies = matching_config.get("matching_strategies", {}) if isinstance(matching_config, dict) else {}
        if not strategies:
            return pipeline_code

        def _sanitize_weights(raw_weights: list[float]) -> list[float]:
            eps = 0.01
            cleaned = [max(float(w), eps) for w in raw_weights]
            total = sum(cleaned)
            if total <= 0:
                return cleaned
            return [round(w / total, 4) for w in cleaned]

        code = pipeline_code
        for pair_key, cfg in strategies.items():
            weights = cfg.get("weights") if isinstance(cfg, dict) else None
            if not weights:
                continue
            weights = _sanitize_weights(weights)
            var = f"weights_{pair_key}"
            weights_literal = "[" + ", ".join([str(float(w)) for w in weights]) + "]"
            pattern = rf"({var}\s*=\s*)\[[^\]]*\]"
            code = re.sub(pattern, rf"\1{weights_literal}", code, count=1)
        return code

    def _normalize_correspondence_saving(self, pipeline_code: str) -> str:
        """Normalize correspondence saving so refined outputs are persisted safely."""
        import re

        code = pipeline_code

        # Fix refined_* usage in to_csv to keep filenames stable
        corr_vars = sorted(set(re.findall(r"\b(?:rb|ml)_correspondences_[A-Za-z0-9_]+\b", code)))
        for var in corr_vars:
            code = code.replace(f"refined_{var}.to_csv", f"{var}.to_csv")

        # Move post-clustering block before saving correspondences if it appears after
        post_match = re.search(
            r"(?ms)^# -{3,}\n# Post-Clustering\n# -{3,}\n.*?(?=^# -{3,}\n# Data Fusion|^# Data Fusion|^# Save Correspondences|\Z)",
            code,
        )
        save_idx = code.find("# Save Correspondences")
        if save_idx == -1:
            save_idx = code.find("# Save correspondences")
        if save_idx == -1:
            save_idx = code.find("CORR_DIR")
        if post_match and save_idx != -1 and post_match.start() > save_idx:
            post_block = code[post_match.start():post_match.end()]
            code = code[:post_match.start()] + code[post_match.end():]
            save_idx = code.find("# Save Correspondences")
            if save_idx == -1:
                save_idx = code.find("# Save correspondences")
            if save_idx == -1:
                save_idx = code.find("CORR_DIR")
            if save_idx != -1:
                code = code[:save_idx] + post_block + "\n" + code[save_idx:]

        # Clean correspondences directory before saving
        if "CORR_DIR" in code and "shutil.rmtree" not in code:
            if "import shutil" not in code:
                if "import os\n" in code:
                    code = code.replace("import os\n", "import os\nimport shutil\n", 1)
                else:
                    code = "import shutil\n" + code
            code = re.sub(
                r"(?m)^os\.makedirs\(CORR_DIR, exist_ok=True\)",
                "if os.path.exists(CORR_DIR):\n    shutil.rmtree(CORR_DIR)\nos.makedirs(CORR_DIR, exist_ok=True)",
                code,
                count=1,
            )

        return code

    def _log_action(self, step: str, action: str, why: str, improvement: str = "", details: dict | None = None):
        payload = {
            "step": step,
            "action": action,
            "why": why,
            "improvement": improvement,
        }
        if details:
            payload["details"] = details
        self.logger.info(json.dumps(payload, ensure_ascii=False))

    def _cluster_analysis_next_step(self, state: SimpleModelAgentState) -> str:
        if state.get("matching_config_updated"):
            return "pipeline_adaption"
        return "evaluation_adaption"

    def should_continue_research(self, state: SimpleModelAgentState) -> str:
        """Determines whether to continue the research loop or end."""
        messages = state['messages']
        last_message = messages[-1]
        # If the LLM made a tool call, continue the loop to process it.
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        # If the LLM responded with code (no tool call), end the loop.
        else:
            return "end"

    # Creates tool calls to profile the data and saves it into agent state 
    def match_schemas(self, state: SimpleModelAgentState):
        self._log_action("match_schemas", "start", "Align dataset schemas before blocking/matching", "Improves comparator compatibility", {"datasets": state.get("datasets")})

        self.logger.info("----------------------- Entering match_schemas -----------------------")
        if state.get("schema_correspondences"):
            print("[*] Schema correspondences already present; skipping match_schemas")
            return {}

        print("[*] Running schema matching")
        result = run_schema_matching(
            dataset_paths=state["datasets"],
            model=self.base_model,
            output_dir="output/schema-matching",
        )
        self.logger.info("Leaving match_schemas")
        return result

    def profile_data(self, state:SimpleModelAgentState):
        self._log_action("profile_data", "start", "Profile datasets to guide feature/threshold choices", "Improves matching signal selection", {"datasets": state.get("datasets")})

        self.logger.info('----------------------- Entering profile_data -----------------------')

        print("[*] Profiling datasets")

        system_prompt = """
            You are a data scientist tasked with the integration of several datasets.
            For each dataset path provided, call the tool `profile_dataset` with the path
            (one tool call per dataset).
        """
        
        datasets_list_str = "\n".join(state['datasets'])
        human_content = f"Please profile these datasets (one call per dataset):\n{datasets_list_str}"
        message = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]
        self.logger.info("Input Message:" + str(message))
        
        result = self._invoke_with_usage(message, "profile_data")
        self.logger.info("RESULT:" + str(result))

        # call tools
        tool_calls = result.tool_calls

        self.logger.info("Tool Calls:" + str(tool_calls))
        results = {}
        for t in tool_calls:
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                self.logger.info("adapt_pipeline: ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            
#            if USE_LLM == "groq" or USE_LLM == "gpt" or USE_LLM == "gemini":
            results[t['args']['path']] = result
#            elif USE_LLM == "gemini_broken":
#                results[t['args']['__arg1']] = result

        with open(OUTPUT_DIR + "profile/profiles.json", "w") as file:
            file.write(json.dumps(results, indent=2))
          
        self.logger.info('Leaving profile_data')
        return {'data_profiles': results}

    def run_blocking_tester(self, state: SimpleModelAgentState):
        self._log_action("run_blocking_tester", "start", "Select blocking strategy to reduce candidates", "Improves runtime and recall balance", {"skip": SKIP_BLOCKING_TESTER})

        self.logger.info("----------------------- Entering run_blocking_tester -----------------------")

        if SKIP_BLOCKING_TESTER:
            path = Path(OUTPUT_DIR + "blocking-evaluation/blocking_config.json")
            if path.exists():
                with path.open("r", encoding="utf-8") as f:
                    blocking_config = json.load(f)
                    print("[+] Using saved blocking strategy: " + json.dumps(blocking_config, indent=2))
                    return {"blocking_config": blocking_config}
            else:
                print("[-] Cannot skip blocking tester. No saved blocking strategy found")
        
        if state.get("blocking_config"):
            print("[*] Blocking config already present in state; skipping BlockingTester")
            return {}

        print("[*] Running BlockingTester")
        tester = BlockingTester(
            llm=self.base_model,
            datasets=state["datasets"],
            max_candidates=350000,
            blocking_testsets=state['entity_matching_testsets'],
            output_dir="output/blocking-evaluation",
            pc_threshold=0.9,
            max_attempts=5,
            max_error_retries=2,
            verbose=True
        )
        _, blocking_config = tester.run_all_pairs()
        print("[*] Blocking config computed:\n" + json.dumps(blocking_config, indent=2))
        self.logger.info("Leaving run_blocking_tester")
        return {"blocking_config": blocking_config}

    def run_matching_tester(self, state: SimpleModelAgentState):
        self._log_action("run_matching_tester", "start", "Select matching strategy and thresholds", "Improves correspondence quality", {"skip": SKIP_MATCHING_TESTER, "matcher_mode": state.get("matcher_mode")})

        self.logger.info("----------------------- Entering run_matching_tester -----------------------")

        if SKIP_MATCHING_TESTER:
            path = Path(OUTPUT_DIR + "matching-evaluation/matching_config.json")
            if path.exists():
                with path.open("r", encoding="utf-8") as f:
                    matching_config = json.load(f)
                    print("[+] Using saved matching strategy: " + json.dumps(matching_config, indent=2))
                    return {"matching_config": matching_config}
            else:
                print("[-] Cannot skip matching tester. No saved matching strategy found")
        
        if state.get("matching_config"):
            print("[*] Matching config already present in state; skipping MatchingTester")
            return {}

        print("[*] Running MatchingTester")
        tester = MatchingTester(
            llm=self.base_model,
            datasets=state["datasets"],
            matching_testsets=state['entity_matching_testsets'],
            blocking_config=state.get("blocking_config", {}),
            output_dir="output/matching-evaluation",
            f1_threshold=0.75,
            max_attempts=4,
            max_error_retries=2,
            verbose=True,
            matcher_mode=state.get("matcher_mode", "ml"),
        )
        _, matching_config = tester.run_all()
        print("[*] Matching config computed:\n" + json.dumps(matching_config, indent=2))
        self.logger.info("Leaving run_matching_tester")
        return {"matching_config": matching_config}


    def pipeline_adaption(self, state: SimpleModelAgentState):
        self._log_action("pipeline_adaption", "start", "Generate integration pipeline code", "Incorporates configs and prior feedback")

        self.logger.info('----------------------- Entering pipeline_adaption -----------------------')

        ####### PREPARE SYSTEM PROMT #######

        # Sync matching_config from disk so updated thresholds carry over
        matching_path = os.path.join(OUTPUT_DIR, "matching-evaluation", "matching_config.json")
        if os.path.exists(matching_path):
            with open(matching_path, "r", encoding="utf-8") as f:
                state["matching_config"] = json.load(f)
    
        # Load example pipeline code
        matcher_mode = state.get("matcher_mode", "ml")
        example_name = "example_pipeline_ml.py" if matcher_mode == "ml" else "example_pipeline.py"
        example_pipeline_path = os.path.join(INPUT_DIR, f"example_pipelines/{example_name}")
        if not os.path.exists(example_pipeline_path):
            raise FileNotFoundError(f"Example pipeline not found at: {example_pipeline_path}")
    
        with open(example_pipeline_path, "r", encoding="utf-8") as f:
            example_pipeline_code = f.read()

        # Prepare first-row previews for each dataset
        dataset_previews = {}
        for path in state['datasets']:
            df = load_dataset(path)
            if df is not None and not df.empty:
                # Take only first row and convert to dictionary
                dataset_previews[path] = df.iloc[0].to_dict()
            else:
                dataset_previews[path] = "Failed to load or empty dataset"
                
        # Prepare prompt for the model
        entity_matching_section = ""

        if state["matcher_mode"] == "ml":
            entity_matching_section = f"""
            2b. Entity matching testsets paths:
            {state["entity_matching_testsets"]}
            """
        
        system_prompt = f"""
        You are a data scientist tasked with the integration of several datasets.
        You are provided with the following inputs:

        1. An example integration pipeline (Python code using PyDI library). Pay
        attention to the comments within the pipeline, as they also contain important instructions:
        {example_pipeline_code}
    
        2. A list of dataset file paths:
        {json.dumps(state['datasets'], indent=2)}

        {entity_matching_section}
    
        3. The first row of each dataset to help understand the structure:
        {json.dumps(dataset_previews, indent=2)}
    
        4. A dictionary containing the profile data of the datasets
        (including number of rows, nulls_per_column and dtypes of 
        the columns):
        {json.dumps(state['data_profiles'], indent=2)}
        """
        
        # Include blocking config if available (from BlockingTester)
        blocking_config = state.get('blocking_config')
        if blocking_config:
            system_prompt += f"""

        5. **BLOCKING CONFIGURATION** (pre-computed optimal blocking strategies):
        This configuration was determined by a blocking evaluation agent. Use these settings
        for your blocking step in the pipeline:
        {json.dumps(blocking_config, indent=2)}
        
        IMPORTANT: Use the id_columns and blocking_strategies from this config:
        - Use the correct id_column for each dataset as specified
        - Use the recommended strategy (exact_match_single/multi or semantic_similarity)
        - Use the recommended columns for blocking
        - Strategy to blocker mapping:
          * exact_match_single / exact_match_multi -> StandardBlocker (exact match on columns)
          * token_blocking / ngram_blocking -> TokenBlocker (token or n-gram blocking)
          * sorted_neighbourhood -> SortedNeighbourhoodBlocker (window)
          * semantic_similarity -> EmbeddingBlocker (top_k)
        """

        # Include matching config if available (from MatchingTester)
        matching_config = state.get('matching_config')
        if matching_config:
            if matcher_mode == "ml":
                system_prompt += f"""

        6. **MATCHING CONFIGURATION** (pre-computed comparator settings):
        This configuration was determined by a matching evaluation agent. Use these settings
        for your MLBasedMatcher feature extraction and training:
        {json.dumps(matching_config, indent=2)}

        IMPORTANT: Use the matching_strategies from this config:
        - Build comparators (StringComparator/NumericComparator/DateComparator) for each dataset pair
        - Use the comparators as features in FeatureExtractor
        - Train a classifier on the labeled pairs (labels in the testset) and use MLBasedMatcher
        - Do NOT set weights; ML model learns weights internally
        - Do NOT add RuleBasedMatcher fallback branches
        - Follow the example ML pipeline structure and naming closely; do not invent new variable roles
        - preprocess mapping: "lower" -> str.lower, "strip" -> str.strip, "lower_strip" -> lambda x: str(x).lower().strip()
        """
            else:
                system_prompt += f"""

        6. **MATCHING CONFIGURATION** (pre-computed comparator settings):
        This configuration was determined by a matching evaluation agent. Use these settings
        for your RuleBasedMatcher step in the pipeline:
        {json.dumps(matching_config, indent=2)}

        IMPORTANT: Use the matching_strategies from this config:
        - Build comparators (StringComparator/NumericComparator/DateComparator) for each dataset pair
        - Use the specified weights and threshold.
        - For each pair in `matching_strategies`, you **MUST** define a
            variable named `threshold_[original_pair_key]` (e.g.,
            `threshold_discogs_lastfm = 0.7`) with the corresponding threshold value.
            Ensure your RuleBasedMatcher instances use these variables for their
            `threshold` parameter.
        - preprocess mapping: "lower" -> str.lower, "strip" -> str.strip, "lower_strip" -> lambda x: str(x).lower().strip()
        """

        system_prompt += """

        Your task is to **create a similar integration pipeline** so that it works with
        the datasets provided. Output should only consist of the relevant Python code
        for the integration pipeline.
        """

        evaluation_analysis = state.get("evaluation_analysis", None)
        if evaluation_analysis:
            system_prompt += f"""
            8. Evaluation reasoning from prior pipeline run:
            {evaluation_analysis}
            """
        #### Cluster Analysis ####
        cluster_analysis = state.get("cluster_analysis_result")
        # cluster_recommendation = None
        # cluster_parameters = {}
        # cluster_recommendation_source = None

        # applied_recommendation = state.get("cluster_recommendation_applied")

        if cluster_analysis:
            # overall = cluster_analysis.get("_overall", {}) if isinstance(cluster_analysis, dict) else {}
            # if isinstance(overall, dict) and overall.get("recommended_strategy") not in (None, "", "None"):
            #     cluster_recommendation = overall.get("recommended_strategy")
            #     cluster_parameters = overall.get("parameters", {})
            #     cluster_recommendation_source = "_overall"
            # else:
            #     for key, value in (cluster_analysis or {}).items():
            #         if not isinstance(value, dict):
            #             continue
            #         rec = value.get("recommended_strategy")
            #         if rec and rec != "None":
            #             cluster_recommendation = rec
            #             cluster_parameters = value.get("parameters", {})
            #             cluster_recommendation_source = key
            #             break

            system_prompt += f"""

            **CLUSTER ANALYSIS FEEDBACK FROM PREVIOUS RUN:**
            Report: {json.dumps(cluster_analysis, indent=2)}

            CLUSTER ANALYSIS: The report includes deep diagnostics (large clusters, hub entities, attribute agreement).
            Use this feedback to guide **matching configuration updates only**. Do **not** add post-clustering steps.
            """

        # Determine if this is initial generation, fix due to execution error, or fix due to poor evaluation
        if "pipeline_execution_result" not in state and "evaluation_metrics" not in state:
            print("[*] Creating initial pipeline")
            human_content = "Create the integration pipeline for the provided datasets."
    
        else:
            # Either a pipeline execution error or evaluation-based adaption
            broken_pipeline_path = OUTPUT_DIR + "code/pipeline.py"
            with open(broken_pipeline_path, "r", encoding="utf-8", errors="ignore") as f:
                broken_code = f.read()
    
            human_content = "You previously generated Python integration pipeline code.\n"
            
            # Add execution error if exists
            if "pipeline_execution_result" in state and state["pipeline_execution_result"].lower().startswith("error"):
                print("[*] Trying to fix pipeline")
                human_content += f"Executing this pipeline caused the following error:\n{state['pipeline_execution_result']}\n"
    
            # Add evaluation feedback if available
            if "evaluation_metrics" in state:
                print("[*] Trying to improve pipeline based on evaluation")
                human_content += f"Evaluation of the last run shows the following metrics:\n{json.dumps(state['evaluation_metrics'], indent=2)}\n"
                human_content += "Improve the pipeline to increase overall accuracy and correct errors highlighted by the evaluation.\n"
    
            human_content += "Here is the current pipeline code:\n" + broken_code
            human_content += "\nOutput ONLY the corrected Python code."
            
        force_refresh = bool(state.get("matching_config_updated") or state.get("matching_config_update_notes"))

        ####### SEARCH DOCUMENTATION TOOL #######
        if "search_documentation" in self.tools and "pipeline_execution_result" not in state.get('messages', []):
             self.tools["search_documentation"].reset()

        if force_refresh or not state.get('messages'):
            # Initial prompt for the research and adaption task
            system_prompt += """
            **PROCESS:**
            1.  **THINK**: Analyze the provided data profiles, configurations, and any previous error reports.
            2.  **RESEARCH**: If you are unsure how to use a PyDI function or class, you MUST use the `search_documentation` tool. You can call it multiple times based on given information such as, blocking configurations, matching configurations, data profile etc. Ask specific questions (e.g., "How to use SortedNeighbourhoodBlocker?", "What are the parameters for RuleBasedMatcher?").
            3.  **CODE**: Once you have gathered enough information, write the complete, executable Python code for the pipeline. **Your final output in this process must be only the Python code itself.**"""
        
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=human_content)
            ]
        else:
            messages = state["messages"]
    
        self.logger.info("Input Message:" + str(messages))
    
        # Call the model
        adapted_pipeline = self._invoke_with_usage(messages, "pipeline_adaption")
        messages.append(adapted_pipeline)

        if adapted_pipeline.tool_calls:
            tool_messages = []
            for tool_call in adapted_pipeline.tool_calls:
                tool_name = tool_call["name"]
                if tool_name in self.tools:
                    tool_output = self.tools[tool_name].invoke(tool_call["args"])
                    tool_messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))
            messages.extend(tool_messages)
            
            # After tool calls, invoke the model again to get the final code
            final_response = self._invoke_with_usage(messages, "pipeline_adaption")
            messages.append(final_response)
            adapted_pipeline = final_response
        
        # Extract the code from the final response
        if hasattr(adapted_pipeline, 'tool_calls') and adapted_pipeline.tool_calls:
            # If still has tool calls after second invocation, recursively handle
            adapted_pipeline = "Pipeline code not available - too many tool calls"
        else:
            adapted_pipeline = adapted_pipeline.content if hasattr(adapted_pipeline, "content") else str(adapted_pipeline)
            self.logger.info("RESULT:" + str(adapted_pipeline))
            if state.get("matching_config"):
                adapted_pipeline = self._apply_matching_weights(
                    adapted_pipeline,
                    state.get("matching_config"),
                )
            applied_recommendation = state.get("cluster_recommendation_applied")
            adapted_pipeline = self._normalize_correspondence_saving(adapted_pipeline)
            # if state.get("matching_config"):
                # adapted_pipeline = self._apply_matching_thresholds(adapted_pipeline, state.get("matching_config", dict()))
            if adapted_pipeline.startswith("```python") and adapted_pipeline.endswith("```"):
                adapted_pipeline = adapted_pipeline.strip("```python").strip("```").strip()
                if state.get("matching_config"):
                    os.makedirs(OUTPUT_DIR + "code/", exist_ok=True)
                    with open(OUTPUT_DIR + "code/pipeline.py", 'w', errors="ignore") as file:
                        file.write(str(adapted_pipeline))
                    print("[+] Applied programmatic threshold updates to generated pipeline code.")
                
                os.makedirs(OUTPUT_DIR + "code/", exist_ok=True)
                with open(OUTPUT_DIR + "code/pipeline.py", 'w', errors="ignore") as file:
                    file.write(str(adapted_pipeline))
                    # self._rewrite_pipeline_thresholds(OUTPUT_DIR + "code/pipeline.py", state.get("matching_config", {}))
            elif not adapted_pipeline.startswith("Pipeline code not available"):
                # Code without markdown markers - still save it
                os.makedirs(OUTPUT_DIR + "code/", exist_ok=True)
                with open(OUTPUT_DIR + "code/pipeline.py", 'w', errors="ignore") as file:
                    file.write(str(adapted_pipeline))
                    # self._rewrite_pipeline_thresholds(OUTPUT_DIR + "code/pipeline.py", state.get("matching_config", {}))

        self.logger.info('Leaving pipeline_adaption')
        return {
            "messages": messages,
            "integration_pipeline_code": adapted_pipeline,
            "cluster_recommendation_applied": applied_recommendation,
            "matching_config_updated": False,
        }

    def execute_pipeline(self, state: SimpleModelAgentState):
        self._log_action("execute_pipeline", "start", "Run generated pipeline", "Produces correspondences for analysis", {"attempt": state.get("pipeline_execution_attempts", 0)+1})

        self.logger.info('----------------------- Entering execute_pipeline -----------------------')

        print("[*] Running generated pipeline")

        attempts = state.get("pipeline_execution_attempts", 0) + 1
        state["pipeline_execution_attempts"] = attempts
    
        pipeline_path = OUTPUT_DIR + "code/pipeline.py"
    
        try:
            result = subprocess.run(
                [sys.executable, pipeline_path],
                capture_output=True,
                text=True,
                timeout=3600
            )
    
            if result.returncode == 0:
                print("[+] Pipeline successfully completed")
                execution_result = "success"
            else:
                execution_result = f"error: {result.stderr}"[:10000]
                print("Error: " + str(execution_result))
    
        except Exception as e:
            execution_result = f"error: {str(e)}"
    
        self.logger.info("Pipeline execution result: " + execution_result)
        self.logger.info('Leaving execute_pipeline')
    
        return {
            "pipeline_execution_result": execution_result,
            "pipeline_execution_attempts": attempts
        }
    
    def cluster_analysis(self, state: SimpleModelAgentState) -> Dict[str, Any]:
        self._log_action("cluster_analysis", "start", "Analyze clusters for over/under-matching", "Guides next-iteration improvements")

        """
        Runs cluster analysis and gets specific recommendations for post-processing.
        """
        self.logger.info("----------------------- Entering cluster_analysis -----------------------")
        print("[*] Running Cluster Analysis to get recommendations...")

        correspondences_dir = "output/correspondences/"
        if not os.path.exists(correspondences_dir):
            print(f"[-] Correspondences directory not found, skipping cluster analysis.")
            return {"cluster_analysis_result": {}}

        current_dataset_names = [Path(p).stem for p in state["datasets"]]
        expected_pairs = set()
        for pair in combinations(current_dataset_names, 2):
            sorted_pair = tuple(sorted(pair))
            expected_pairs.add(f"{sorted_pair[0]}_{sorted_pair[1]}")
            expected_pairs.add(f"{sorted_pair[1]}_{sorted_pair[0]}")

        all_files = os.listdir(correspondences_dir)
        correspondence_files_to_process = []
        for f in all_files:
            if f.endswith(".csv") and any(pair in f for pair in expected_pairs):
                correspondence_files_to_process.append(os.path.join(correspondences_dir, f))
        
        if not correspondence_files_to_process:
            print(f"[-] No relevant correspondence files found, skipping cluster analysis.")
            return {"cluster_analysis_result": {}}
        
        print(f"[*] Found {len(correspondence_files_to_process)} relevant correspondence files to analyze.")

        try:
            # Use the ClusterTester
            dataset_paths = {Path(p).stem: p for p in state.get("datasets", [])}
            id_columns = (state.get("blocking_config", {}) or {}).get("id_columns", {})
            cluster_tester = ClusterTester(
                llm=self.base_model,
                verbose=True,
                dataset_paths=dataset_paths,
                id_columns=id_columns,
            )
            report = cluster_tester.run(correspondence_files_to_process)

            matching_path = os.path.join(OUTPUT_DIR, "matching-evaluation", "matching_config.json")
            matching_config = None
            if os.path.exists(matching_path):
                with open(matching_path, "r", encoding="utf-8") as f:
                    matching_config = json.load(f)
            matching_config_before = json.loads(json.dumps(matching_config)) if matching_config else None

            target_thresholds = (report.get("_overall", {}) or {}).get("target_thresholds", {})
            target_small = float(target_thresholds.get("small_cluster_ratio", 0.9))
            target_large = float(target_thresholds.get("large_cluster_ratio", 0.005))
            target_one_to_many = float(target_thresholds.get("one_to_many_ratio", 0.03))

            history_path = os.path.join("output/cluster-evaluation", "cluster_analysis_history.json")
            history_payload = []
            if os.path.exists(history_path):
                try:
                    with open(history_path, "r", encoding="utf-8") as f:
                        history_payload = json.load(f) or []
                except Exception:
                    history_payload = []

            max_update_attempts = 2
            if isinstance(matching_config, dict):
                try:
                    max_update_attempts = int(matching_config.get("_max_update_attempts", max_update_attempts))
                except Exception:
                    max_update_attempts = 2

            update_attempts = {}
            for item in history_payload:
                if not isinstance(item, dict):
                    continue
                update_info = item.get("_matching_config_update", {}) if isinstance(item, dict) else {}
                if not update_info.get("updated"):
                    continue
                changes = update_info.get("changes", {}) or {}
                for pair_key in changes.keys():
                    update_attempts[pair_key] = update_attempts.get(pair_key, 0) + 1

            config_attempts = {}
            if isinstance(matching_config, dict):
                config_attempts = matching_config.get("_update_attempts", {}) or {}
            for pair_key, count in config_attempts.items():
                try:
                    count_int = int(count)
                except Exception:
                    count_int = 0
                update_attempts[pair_key] = max(update_attempts.get(pair_key, 0), count_int)

            def _meets_target(entry: dict) -> bool:
                health = (entry or {}).get("health_metrics", {}) or {}
                small = health.get("small_cluster_ratio")
                large = health.get("large_cluster_ratio")
                otm = health.get("one_to_many_ratio")
                if small is None or large is None or otm is None:
                    return False
                return small >= target_small and large <= target_large and otm <= target_one_to_many

            def _pair_key_from_report(name: str) -> str:
                stem = Path(name).stem
                if stem.startswith("correspondences_"):
                    stem = stem[len("correspondences_"):]
                return stem

            pair_targets = {}
            pairs_to_update = []
            for name, entry in (report or {}).items():
                if not isinstance(entry, dict) or name.startswith("_"):
                    continue
                pair_key = _pair_key_from_report(name)
                computed_meets_target = _meets_target(entry)
                meets_target = bool(entry.get("meets_target") or computed_meets_target)
                if computed_meets_target:
                    entry["meets_target"] = True
                    entry["recommended_action"] = "None"
                recommended_action = entry.get("recommended_action")
                pair_targets[pair_key] = {
                    "meets_target": meets_target,
                    "recommended_action": recommended_action,
                    "health_metrics": entry.get("health_metrics"),
                    "duplicate_stats": entry.get("duplicate_stats"),
                    "score_stats": entry.get("score_stats"),
                    "update_attempts": update_attempts.get(pair_key, 0),
                }
                if (not meets_target) and entry.get("health_metrics"):
                    if update_attempts.get(pair_key, 0) < max_update_attempts:
                        pairs_to_update.append(pair_key)

            pairs_to_update = sorted(set(pairs_to_update))

            skipped_due_to_max = [
                pair_key
                for pair_key, meta in pair_targets.items()
                if (not meta.get("meets_target"))
                and update_attempts.get(pair_key, 0) >= max_update_attempts
            ]

            if isinstance(report, dict) and report.get("_overall"):
                report["_overall"]["recommended_action"] = ("update_matching_config" if pairs_to_update else "None")

            # LLM-driven matching_config update based on cluster analysis history
            matching_config_updated = False
            matching_update_notes = ""
            if matching_config and isinstance(report, dict):
                overall_action = report.get("_overall", {}).get("recommended_action")
                if overall_action != "update_matching_config" or not pairs_to_update:
                    overall_action = None
                system_prompt = """
                    You are a data integration expert. Given cluster analysis history and the current matching_config.json,
                    analyze the data and think rationally to propose precise updates to matcher weights to reduce large clusters and one-to-many matches.

                    Rules:
                    - Only edit matching_strategies.*.weights.
                    - Only update the pairs listed in pairs_to_update. Do not include other pairs.
                    - Keep the number of weights equal to the number of comparators for that pair.
                    - Keep weights non-negative and normalize them to sum to 1.
                    - The individual weight values must be between 0.01 and 0.99 to avoid complete elimination or over-reliance on a single comparator.
                    - Do NOT change thresholds, comparators, columns, or strategy names.

                    Return strict JSON with this schema:
                    {
                    "matching_strategies": {
                        "pair_key": {
                        "weights": [..],
                        "rationale": "short"
                        }
                    },
                    "notes": "short"
                    }
                    """
                payload = {
                    "cluster_analysis_latest": report,
                    "cluster_analysis_history": history_payload[-3:],
                    "matching_config": matching_config,
                    "pairs_to_update": pairs_to_update,
                    "pair_metrics": pair_targets,
                    "update_attempts": update_attempts,
                    "max_update_attempts": max_update_attempts,
                }
                messages = [
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=json.dumps(payload, indent=2)),
                ]
                try:
                    if not overall_action:
                        if skipped_due_to_max:
                            matching_update_notes = (
                                "Max update attempts reached for: " + ", ".join(sorted(skipped_due_to_max))
                            )
                        else:
                            matching_update_notes = "All pairs meet target thresholds; skipping weight update."
                    else:
                        llm_result = self.base_model.invoke(messages)
                        raw_text = llm_result.content if hasattr(llm_result, "content") else str(llm_result)
                        raw_text = raw_text.strip().strip("```json").strip("```")
                        proposal = json.loads(raw_text)
                        updates = proposal.get("matching_strategies", {}) if isinstance(proposal, dict) else {}
                        changed = False
                        updated_pairs = []
                        for pair_key, update in updates.items():
                            if pair_key not in pairs_to_update:
                                continue
                            if pair_key not in matching_config.get("matching_strategies", {}):
                                continue
                            strategy = matching_config["matching_strategies"][pair_key]
                            weights = update.get("weights") if isinstance(update, dict) else None
                            if weights and isinstance(weights, list) and len(weights) == len(strategy.get("comparators", [])):
                                eps = 0.01
                                cleaned = [max(float(w), eps) for w in weights]
                                total = sum(cleaned)
                                if total > 0:
                                    weights = [round(w / total, 4) for w in cleaned]
                                    strategy["weights"] = weights
                                    changed = True
                                    updated_pairs.append(pair_key)
                        if changed:
                            attempts_store = matching_config.get("_update_attempts", {}) if isinstance(matching_config, dict) else {}
                            for pair_key in updated_pairs:
                                try:
                                    current = int(attempts_store.get(pair_key, 0))
                                except Exception:
                                    current = 0
                                attempts_store[pair_key] = current + 1
                            matching_config["_update_attempts"] = attempts_store
                            matching_config_updated = True
                            matching_update_notes = proposal.get("notes", "") if isinstance(proposal, dict) else ""
                            os.makedirs(os.path.dirname(matching_path), exist_ok=True)
                            with open(matching_path, "w", encoding="utf-8") as f:
                                json.dump(matching_config, f, indent=2)

                            # Print before/after weights per updated pair
                            if matching_config_before:
                                print("[*] Matching config updated (weights only):")
                                for pair_key in updated_pairs:
                                    if pair_key not in matching_config_before.get("matching_strategies", {}):
                                        continue
                                    before_weights = matching_config_before["matching_strategies"][pair_key].get("weights")
                                    after_weights = matching_config["matching_strategies"][pair_key].get("weights")
                                    if before_weights != after_weights:
                                        print(f"    - {pair_key}: {before_weights} -> {after_weights}")
                except Exception:
                    matching_config_updated = False

            comparison_prev = {}
            history = state.get("cluster_analysis_history") or []
            history_path = os.path.join("output/cluster-evaluation", "cluster_analysis_history.json")
            if not history and os.path.exists(history_path):
                try:
                    with open(history_path, "r", encoding="utf-8") as f:
                        history = json.load(f) or []
                except Exception:
                    history = []
            if history:
                prev_report = history[-1]
                def _metrics(report: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
                    metrics = {}
                    for name, entry in (report or {}).items():
                        if not isinstance(entry, dict) or name.startswith("_"):
                            continue
                        health = entry.get("health_metrics", {}) or {}
                        metrics[name] = {
                            "total_clusters": health.get("total_clusters"),
                            "max_cluster_size": health.get("max_cluster_size"),
                            "small_cluster_ratio": health.get("small_cluster_ratio"),
                            "large_cluster_ratio": health.get("large_cluster_ratio"),
                            "one_to_many_ratio": health.get("one_to_many_ratio"),
                        }
                    return metrics

                prev_metrics = _metrics(prev_report)
                curr_metrics = _metrics(report)
                for name in set(prev_metrics) | set(curr_metrics):
                    before = prev_metrics.get(name, {})
                    after = curr_metrics.get(name, {})
                    comparison_prev[name] = {
                        "before": before,
                        "after": after,
                        "delta": {
                            key: (
                                None
                                if before.get(key) is None or after.get(key) is None
                                else round(after.get(key) - before.get(key), 4)
                            )
                            for key in {"total_clusters", "max_cluster_size", "small_cluster_ratio", "large_cluster_ratio", "one_to_many_ratio"}
                        },
                    }

            if history:
                before_snapshot = {
                    k: v
                    for k, v in prev_report.items()
                    if k not in ("_comparison_prev", "_before")
                }
                report["_before"] = before_snapshot

            if matching_config_updated:
                update_summary = {}
                for pair_key, strat in (matching_config_before or {}).get("matching_strategies", {}).items():
                    if pair_key not in matching_config.get("matching_strategies", {}):
                        continue
                    before_w = strat.get("weights")
                    after_w = matching_config["matching_strategies"][pair_key].get("weights")
                    if before_w != after_w:
                        update_summary[pair_key] = {
                            "weights_before": before_w,
                            "weights_after": after_w,
                        }
                report["_matching_config_update"] = {
                    "updated": True,
                    "changes": update_summary,
                    "notes": matching_update_notes,
                    "pairs_to_update": pairs_to_update,
                    "skipped_pairs": sorted(set(pair_targets) - set(pairs_to_update)),
                    "skipped_due_to_max": sorted(skipped_due_to_max),
                    "max_update_attempts": max_update_attempts,
                    "update_attempts": update_attempts,
                }
            else:
                report["_matching_config_update"] = {
                    "updated": False,
                    "changes": {},
                    "notes": matching_update_notes,
                    "pairs_to_update": pairs_to_update,
                    "skipped_pairs": sorted(set(pair_targets) - set(pairs_to_update)),
                    "skipped_due_to_max": sorted(skipped_due_to_max),
                    "max_update_attempts": max_update_attempts,
                    "update_attempts": update_attempts,
                }

            report["_comparison_prev"] = comparison_prev
            history.append(
                {k: v for k, v in report.items() if k not in ("_comparison_prev", "_before")}
            )
            os.makedirs(os.path.dirname(history_path), exist_ok=True)
            with open(history_path, "w", encoding="utf-8") as f:
                json.dump(history, f, indent=2)

            # report_path = os.path.join("output/cluster-evaluation", "cluster_analysis_report.json")
            # os.makedirs(os.path.dirname(report_path), exist_ok=True)
            # # Write human-readable summary markdown (LLM-generated)
            # summary_path = os.path.join("output/cluster-evaluation", "cluster_analysis_summary.md")
            # try:
            #     summary_prompt = """
            #     You are a data integration expert. Summarize the provided cluster analysis report into clear, human-readable markdown.

            #     Include these sections:
            #     1) Overall Snapshot
            #     2) What Was Wrong (Before) vs What Happened After (Now) — per correspondence file with key metrics
            #     3) Deep Analysis Highlights (Large Clusters)
            #     4) Conclusion (1-3 sentences)

            #     Be concise, structured, and avoid raw JSON dumps. Use bullet points.
            #     """
            #     summary_input = json.dumps(history, indent=2)
            #     messages = [
            #         SystemMessage(content=summary_prompt),
            #         HumanMessage(content=summary_input),
            #     ]
            #     llm_result = self.base_model.invoke(messages)
            #     summary_content = llm_result.content if hasattr(llm_result, "content") else str(llm_result)
            #     with open(summary_path, "w", encoding="utf-8") as f:
            #         f.write(summary_content)
            # except Exception:
            #     pass

            return {
                "cluster_analysis_result": report,
                "cluster_analysis_history": history,
                "matching_config": matching_config if matching_config else state.get("matching_config"),
                "matching_config_updated": matching_config_updated,
                "matching_config_update_notes": matching_update_notes,
            }
        except Exception as e:
            print(f"[-] Error during cluster analysis: {e}")
            self.logger.error(f"Cluster analysis failed: {traceback.format_exc()}")
            return {"cluster_analysis_result": {"error": str(e)}}
        
    def evaluation_adaption(self, state: SimpleModelAgentState):
        self._log_action("evaluation_adaption", "start", "Generate evaluation code", "Measures fusion quality")

        self.logger.info('----------------------- Entering evaluation_adaption -----------------------')
        
        example_eval_path = INPUT_DIR + "example_pipelines/example_evaluation.py"
        example_eval_code = open(example_eval_path).read()
    
        fused_output_path = "output/data_fusion/fusion_data.csv"

        # Prepare first-row preview for testset
        dataset = load_dataset(state['fusion_testset'])
        if dataset is not None and not dataset.empty:
            # Take only first row and convert to dictionary
            testset_preview = dataset.iloc[0].to_dict()
        else:
            testset_preview = "Failed to load or empty dataset"
    
        system_prompt = f"""
        You are a data scientist evaluating a data fusion pipeline.
    
        Example evaluation code:
        {example_eval_code}
    
        Generated integration pipeline:
        {state['integration_pipeline_code']}
    
        Dataset profiles:
        {json.dumps(state['data_profiles'], indent=2)}
    
        The fused output is located at:
        {fused_output_path}

        The testset is located at:
        {state['fusion_testset']}

        The first row of the testset looks like the following:
        {testset_preview}
    
        Create evaluation code that:
        - Uses the correct fusion strategy
        - Loads the fused output
        - Loads the gold standard test set
        - Prints structured evaluation metrics
        - Prints chosen evaluation functions in a compact one-line summary
        """

        evaluation_analysis = state.get("evaluation_analysis", None)
        if evaluation_analysis:
            system_prompt += f"""
            Evaluation reasoning from prior pipeline run:
            {evaluation_analysis}
            """

        ####### HUMAN PROMPT #######
        if not state.get("evaluation_execution_result"):
            print("[*] Adapting evaluation code")
            # First generation
            human_content = """
            Create the evaluation code.
            """
        else:
            # Fix broken evaluation
            print("[*] Fixing evaluation code")
            attempts = state.get("evaluation_execution_attempts", 0)
    
            if attempts >= 3:
                self.logger.info("Maximum evaluation fix attempts reached")
                return {"evaluation_execution_result": "error: max attempts reached"}
    
            eval_path = OUTPUT_DIR + "code/evaluation.py"
            with open(eval_path, "r", encoding="utf-8") as f:
                broken_code = f.read()
    
            error = state["evaluation_execution_result"]
    
            human_content = f"""
            You previously generated Python evaluation code.
            Executing this code caused the following error:
    
            {error}
    
            Here is the current evaluation code:
            {broken_code}
    
            Fix the code so that it executes successfully.
            Output ONLY the corrected Python code.
            """

        if "search_documentation" in self.tools and not state.get("evaluation_execution_result"):
            self.tools["search_documentation"].reset()

        if not state.get('eval_messages'):
            # Initial prompt for the evaluation adaption task
            system_prompt += """
            **PROCESS:**
            1.  **THINK**: Analyze the provided integration pipeline code, dataset profiles, and any previous error reports.
            2.  **RESEARCH**: If you are unsure how to use a PyDI function or class for evaluation, you MUST use the `search_documentation` tool. You can call it multiple times based on given information such as, fusion strategy, data profiles, error messages etc. Ask specific questions (e.g., "How to evaluate a fusion output?", "What functions does PyDI provide for evaluation?").
            3.  **CODE**: Once you have gathered enough information, write the complete, executable Python code for the evaluation. **Your final output in this process must be only the Python code itself.**"""
        
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=human_content)
            ]
            state["eval_messages"] = messages
        else:
            messages = state["eval_messages"]

        self.logger.info("Input Message:" + str(messages))

        result = self._invoke_with_usage(messages, "evaluation_adaption")
        messages.append(result)

        if result.tool_calls:
            tool_messages = []
            for tool_call in result.tool_calls:
                tool_name = tool_call["name"]
                if tool_name in self.tools:
                    tool_output = self.tools[tool_name].invoke(tool_call["args"])
                    tool_messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))
            messages.extend(tool_messages)
            
            # After tool calls, invoke the model again to get the final code
            final_response = self._invoke_with_usage(messages, "evaluation_adaption")
            messages.append(final_response)
            result = final_response

        if hasattr(result, 'tool_calls') and result.tool_calls:
            # If still has tool calls after second invocation, recursively handle
            code = "Evaluation code not available - too many tool calls"
        else:
            code = result.content if hasattr(result, "content") else str(result)
            self.logger.info("RESULT:" + str(code))

            code = code.strip("```python").strip("```")
            with open(OUTPUT_DIR + "code/evaluation.py", "w") as f:
                f.write(code)

        self.logger.info('Leaving evaluation_adaption')
    
        return {"evaluation_code": code, "eval_messages": messages}

    def execute_evaluation(self, state: SimpleModelAgentState):
        self._log_action("execute_evaluation", "start", "Run evaluation", "Produces accuracy metrics", {"attempt": state.get("evaluation_execution_attempts", 0)+1})

        self.logger.info('----------------------- Entering execute_evaluation -----------------------')
    
        print("[*] Running generated evaluation")
    
        attempts = state.get("evaluation_execution_attempts", 0) + 1
        state["evaluation_execution_attempts"] = attempts
    
        evaluation_path = OUTPUT_DIR + "code/evaluation.py"
    
        try:
            result = subprocess.run(
                [sys.executable, evaluation_path],
                capture_output=True,
                text=True,
                timeout=3600
            )
    
            if result.returncode == 0:
                print("[+] Evaluation successfully completed")
                execution_result = "success"
            else:
                print("[-] Evaluation could not be executed")
                execution_result = f"error: {result.stderr}"[:10000]
                print("Error: " + str(execution_result))
    
        except Exception as e:
            execution_result = f"error: {str(e)}"
    
        self.logger.info("Evaluation execution result: " + execution_result)
        self.logger.info('Leaving execute_evaluation')
    
        return {
            "evaluation_execution_result": execution_result,
            "evaluation_execution_attempts": attempts
        }

    def evaluation_decision(self, state: SimpleModelAgentState):
        self._log_action("evaluation_decision", "start", "Decide whether to iterate", "Drives improvement loop")

        self.logger.info('----------------------- Entering evaluation_decision -----------------------')
        print("[*] Reading evaluation metrics")

        attempts = state.get("evaluation_attempts", 0) + 1
    
        eval_path = "output/pipeline_evaluation/pipeline_evaluation.json"
    
        if not os.path.exists(eval_path):
            self.logger.warning("Evaluation file missing")
            metrics = {"error": "evaluation file missing"}
        else:
            with open(eval_path, "r", encoding="utf-8") as f:
                metrics = json.load(f)
    
        self.logger.info(f"Evaluation metrics: {metrics}")
        print(f"Evaluation metrics: {json.dumps(metrics)}")
    
        # Store metrics in the state
        state["evaluation_metrics"] = metrics

        overall_acc = metrics.get("overall_accuracy", 0.0)
        print(f"[*] Overall Accuracy: {overall_acc:.3%}")

        if overall_acc >= 0.85 or attempts >= 3:
            self._print_total_usage()
    
        self.logger.info('Leaving evaluation_decision')

        return {
            "evaluation_metrics": metrics,
            "evaluation_attempts": attempts,
            "pipeline_execution_result": "",
            "pipeline_execution_attempts": 0,
            "evaluation_execution_result": "",
            "evaluation_execution_attempts": 0
        }
    def evaluation_reasoning(self, state: SimpleModelAgentState):
        self._log_action("evaluation_reasoning", "start", "Analyze errors", "Suggests concrete fixes for next run")

        self.logger.info('----------------------- Entering evaluation_reasoning -----------------------')
    
        # Sync matching_config from disk so updated thresholds carry over
        matching_path = os.path.join(OUTPUT_DIR, "matching-evaluation", "matching_config.json")
        if os.path.exists(matching_path):
            with open(matching_path, "r", encoding="utf-8") as f:
                state["matching_config"] = json.load(f)

        debug_path = "output/pipeline_evaluation/debug_fusion_eval.jsonl"
        debug_events = []
    
        if os.path.exists(debug_path):
            with open(debug_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        debug_events.append(json.loads(line))
                    except Exception:
                        continue
    
        # Prepare system prompt
        system_prompt = """
        You are a data integration expert analyzing the evaluation results of a data integration pipeline.
    
        You are given:
        - Aggregate evaluation metrics
        - Detailed per-record debugging events from a fusion evaluation
        - The current integration pipeline code
        - The optimal blocking and matching configurations
        - Detailed Cluster Analysis report with improvement suggestions.
    
        Your task:
        1. Identify the dominant error types (false positives, false negatives, missed blocks, bad matches)
        2. Determine whether errors are more likely caused by:
           - Blocking being too restrictive or too loose
           - Matching thresholds or comparator choices
           - Schema or preprocessing issues
           - Logic in the current pipeline code
        3. Analyze the provided cluster analysis report to suggest improvements.
        4. Suggest concrete, actionable improvements for the next pipeline iteration

        IMPORTANT: The blocking and matching strategies have been determined earlier and have been already evaluated. Therefore, 
        focus on improving the fusion of the pipeline.
    
        Respond with concise, structured reasoning.
        """
    
        # Prepare human content
        human_content = f"""
        Evaluation metrics:
        {json.dumps(state.get("evaluation_metrics", {}), indent=2)}
    
        Debug fusion events (JSONL):
        {json.dumps(debug_events[:50], indent=2)}
    
        Current integration pipeline code:
        {state.get("integration_pipeline_code", "Pipeline code not available")}

        Current pipeline evaluation code:
        {state.get("evaluation_code", "Pipeline evaluation code not available")}
    
        Optimal blocking configuration:
        {json.dumps(state.get("blocking_config", {}), indent=2)}
    
        Optimal matching configuration:
        {json.dumps(state.get("matching_config", {}), indent=2)}

        Cluster Analysis Report:
        {json.dumps(state.get("cluster_analysis_result", {}), indent=2)}
        """
    
        # Model call
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        result = self.model.invoke(message)
        analysis = result.content if hasattr(result, "content") else str(result)
    
        self.logger.info(f"Evaluation reasoning produced\nEvaluation reasoning:\n{analysis}")
        print("[*] Evaluation reasoning:\n" + analysis)

        history = state.get("evaluation_reasoning_history", [])
        history.append({
            "analysis": analysis.replace('"', "'"),
            "evaluation_attempt": state.get("evaluation_attempts", 0),
            "overall_accuracy": state.get("evaluation_metrics", {}).get("overall_accuracy"),
        })
        state["evaluation_reasoning_history"] = history

        return {"evaluation_analysis": analysis, "evaluation_reasoning_history": history}
        
    def save_results(self, state: SimpleModelAgentState):
        self._log_action("save_results", "start", "Persist run artifacts", "Enables reproducibility")

        """Save comprehensive results to timestamped JSON file"""
        self.logger.info('----------------------- Entering save_results -----------------------')


        def _parse_agent_run_log(log_path):
            import re
            """Parse agent_run.log for execution trace, transitions, and step counts."""
            node_counts = {}
            execution_order = []
            log_actions = []
            transitions = []
            last_left_node = None
            if not os.path.exists(log_path):
                return {"node_counts": node_counts, "execution_order": execution_order, "transitions": transitions, "log_actions": log_actions}

            with open(log_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split(" | ", 2)
                    if len(parts) < 3:
                        continue
                    timestamp, level, message = parts[0], parts[1], parts[2]

                    # Track entering lines for sequence
                    if "Entering" in message:
                        m = re.search(r"Entering\s+([A-Za-z0-9_]+)", message)
                        if m:
                            node = m.group(1)
                            execution_order.append({"node": node, "timestamp": timestamp, "event": "enter"})
                            node_counts[node] = node_counts.get(node, 0) + 1
                            if last_left_node:
                                transitions.append({
                                    "current_node": last_left_node,
                                    "next_node": node,
                                    "timestamp": timestamp,
                                })
                                last_left_node = None

                    if "Leaving" in message:
                        m = re.search(r"Leaving\s+([A-Za-z0-9_]+)", message)
                        if m:
                            last_left_node = m.group(1)

                    # Parse structured log actions (JSON payloads)
                    if message.startswith("{"):
                        try:
                            payload = json.loads(message)
                            if isinstance(payload, dict) and "step" in payload and "action" in payload:
                                payload["_timestamp"] = timestamp
                                log_actions.append(payload)
                        except Exception:
                            pass

            return {
                "node_counts": node_counts,
                # "execution_order": execution_order,
                "transitions": transitions,
                "log_actions": log_actions,
            }

        def _extract_query_response_pairs(messages):
            """
            Extract (query, response) pairs from LangChain message objects.
            """
            pairs = []

            # Step 1: collect tool_call_id -> query mapping
            tool_call_map = {}

            for msg in messages:
                if isinstance(msg, AIMessage) and msg.tool_calls:
                    for tool_call in msg.tool_calls:

                        call_id = tool_call.get("id")

                        query = None

                        # Case 1: args stored directly
                        if "args" in tool_call:
                            query = tool_call["args"].get("query")

                        # Case 2: arguments stored as JSON string
                        elif "function" in tool_call:
                            args_str = tool_call["function"].get("arguments", "{}")
                            try:
                                args = json.loads(args_str)
                                query = args.get("query")
                            except json.JSONDecodeError:
                                query = None

                        if call_id and query:
                            tool_call_map[call_id] = query

                # Step 2: match ToolMessage responses
                for msg in messages:
                    if isinstance(msg, ToolMessage):
                        tool_call_id = msg.tool_call_id

                        if tool_call_id in tool_call_map:
                            query = tool_call_map[tool_call_id]
                            response = msg.content

                            pairs.append({
                                "query": query,
                                "response": response
                            })

                return pairs

        # Create timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_dir = os.path.join(OUTPUT_DIR, "results")
        os.makedirs(results_dir, exist_ok=True)

        # Collect dataset information
        dataset_names = [os.path.splitext(os.path.basename(path))[0] for path in state.get("datasets", [])]

        # Collect blocking metrics
        blocking_metrics = {}
        blocking_strategies = state.get("blocking_config", {}).get("blocking_strategies", {})
        for pair_key, config in blocking_strategies.items():
            if isinstance(config, dict):
                blocking_metrics[pair_key] = {
                    "pair_completeness": config.get("pair_completeness", 0),
                    "num_candidates": config.get("num_candidates", 0),
                    "stategy": config.get("strategy", "unknown"),
                    "columns": config.get("columns", [])

                }

        # Collect matching metrics
        matching_metrics = {}
        matching_strategies = state.get("matching_config", {}).get("matching_strategies", {})
        for pair_key, config in matching_strategies.items():
            if isinstance(config, dict):
                matching_metrics[pair_key] = {
                    "f1_score": config.get("f1", 0)
                }

        # Collect evaluation metrics
        evaluation_metrics = state.get("evaluation_metrics", {})

        # Collect pipeline execution info
        pipeline_info = {
            "evaluation_attempts": state.get("evaluation_attempts", 0),
            "matcher_mode": state.get("matcher_mode", "unknown"),
        }

        # Collect data profiles summary
        data_profiles_summary = {}
        data_profiles = state.get("data_profiles", {})
        for path, profile in data_profiles.items():
            if isinstance(profile, dict):
                dataset_name = os.path.splitext(os.path.basename(path))[0]
                data_profiles_summary[dataset_name] = {
                    "num_rows": profile.get("num_rows", 0),
                    "num_columns": profile.get("num_columns", 0),
                    "columns": list(profile.get("dtypes", {}).keys())[:10]  # First 10 columns
                }

        log_summary = _parse_agent_run_log(self.log_path)

        model_name = None
        for candidate in (self.base_model, self.model):
            if candidate is None:
                continue
            for attr in ("model_name", "model", "deployment_name", "name"):
                value = getattr(candidate, attr, None)
                if value:
                    model_name = value
                    break
            if model_name:
                break

        # Compile comprehensive results
        results = {
            "timestamp": timestamp,

            # Basic run information
            "model": model_name,
            "datasets": dataset_names,
            "matcher_mode": state.get("matcher_mode", "unknown"),
            "fusion_testset": state.get("fusion_testset", "").split("/")[-1],

            # Performance metrics
            "blocking_metrics": blocking_metrics,
            "matching_metrics": matching_metrics,
            "evaluation_metrics": evaluation_metrics,
            "overall_accuracy": evaluation_metrics.get("overall_accuracy", 0),

            # Configurations used
            "blocking_config": state.get("blocking_config", {}),
            "matching_config": state.get("matching_config", {}),

            # Token usage
            "token_usage": self.token_usage.copy(),
            "token_usage_by_node": {
                "per_call": self.node_token_usage,
                "totals": self.node_token_usage_totals,
            },

            # Pipeline execution details
            "pipeline_info": pipeline_info,
            "execution_trace": log_summary,

            # Search tool responses
            "pipeline_search_tool_responses": _extract_query_response_pairs(state.get("messages", [])[2:-1]),
            "evaluation_search_tool_responses": _extract_query_response_pairs((state.get("eval_messages") or [])[2:-1]),

            # Cluster analysis
            "cluster_analysis_result": state.get("cluster_analysis_result", {}),

            # Evaluation reasoning history
            "evaluation_reasoning_history": state.get("evaluation_reasoning_history", []),
            "evaluation_analysis_latest": state.get("evaluation_analysis", ""),

            # Code and execution
            "integration_pipeline_code_length": len(state.get("integration_pipeline_code", "")),
            "evaluation_code_length": len(state.get("evaluation_code", "")),

        }

        # Save to timestamped file
        results_file = os.path.join(results_dir, f"run_{timestamp}.json")

        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False, default=str)

        # Write human-readable results summary markdown (LLM-generated)
        summary_dir = os.path.join(OUTPUT_DIR, "results")
        os.makedirs(summary_dir, exist_ok=True)
        summary_path = os.path.join(summary_dir, f"run_{timestamp}.md")
        try:
            summary_prompt = """
            You are a data integration expert. Summarize the run results JSON into clear, human-readable markdown.

            Include these sections:
            1) Run Overview (model, datasets, matcher mode)
            2) Blocking & Matching Highlights
            3) Evaluation Summary
            4) Cluster Analysis Summary
            5) Execution / Token Usage
            6) Evaluation Reasoning Highlights and Suggestions

            Be concise and use bullet points. Avoid raw JSON dumps.
            """
            summary_input = json.dumps(results, indent=2)
            messages = [
                SystemMessage(content=summary_prompt),
                HumanMessage(content=summary_input),
            ]
            llm_result = self.base_model.invoke(messages)
            summary_content = llm_result.content if hasattr(llm_result, "content") else str(llm_result)
            with open(summary_path, "w", encoding="utf-8") as f:
                f.write(summary_content)
        except Exception:
            pass

        print(f"[+] Results saved to: {results_file}")

        self.logger.info(f'Results saved to {results_file}')
        self.logger.info('Leaving save_results')

        return {}

## Invoke Pipeline

In [11]:
if USE_LLM == "gemini":
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
elif USE_LLM == "groq":
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )
elif USE_LLM == "gpt":
    llm = ChatOpenAI(
        model="gpt-5.1",
        temperature=0,
    )

In [12]:
# Music use case
entity_matching_testsets = {
    ("discogs", "lastfm"): INPUT_DIR + "datasets/music/testsets/discogs_lastfm_goldstandard_blocking.csv",
    ("discogs", "musicbrainz"): INPUT_DIR + "datasets/music/testsets/discogs_musicbrainz_goldstandard_blocking.csv",
    ("musicbrainz", "lastfm"): INPUT_DIR + "datasets/music/testsets/musicbrainz_lastfm_goldstandard_blocking.csv",
}
datasets = [
    INPUT_DIR + "datasets/music/discogs.xml",
    INPUT_DIR + "datasets/music/lastfm.xml",
    INPUT_DIR + "datasets/music/musicbrainz.xml"
]
fusion_testset = INPUT_DIR + "datasets/music/testsets/test_set.xml"

In [13]:
# # Restaurants use case
# entity_matching_testsets = {
#     ("kaggle_small", "uber_eats_small"): INPUT_DIR + "datasets/restaurant/testsets/kaggle_uber_eats_goldstandard_blocking_small.csv",
#     ("kaggle_small", "yelp_small"): INPUT_DIR + "datasets/restaurant/testsets/kaggle_yelp_goldstandard_blocking_small.csv",
#     ("uber_eats_small", "yelp_small"): INPUT_DIR + "datasets/restaurant/testsets/uber_eats_yelp_goldstandard_blocking_small.csv",
# }

# datasets = [
#     INPUT_DIR + "datasets/restaurant/kaggle_small.parquet",
#     INPUT_DIR + "datasets/restaurant/uber_eats_small.parquet",
#     INPUT_DIR + "datasets/restaurant/yelp_small.parquet"
# ]
# fusion_testset = INPUT_DIR + "datasets/restaurant/testsets/Restaurant_Fusion_Test_Set.csv"

In [14]:
# # Games use case
# entity_matching_testsets= {
#     ("dbpedia","sales"): INPUT_DIR + "datasets/games/testsets/dpedia_games_sales_goldstandard_blocking.csv",
#     ("dbpedia","metacritic"): INPUT_DIR + "datasets/games/testsets/dpedia_games_metacritic_goldstandard_blocking.csv",
#     ("sales","metacritic"): INPUT_DIR + "datasets/games/testsets/sales_metacritic_goldstandard_blocking.csv",
# }

# datasets= [
#     INPUT_DIR + "datasets/games/dbpedia.xml",
#     INPUT_DIR + "datasets/games/metacritic.xml",
#     INPUT_DIR + "datasets/games/sales.xml"
# ]
# fusion_testset= INPUT_DIR + "datasets/games/testsets/test_set.xml"

In [15]:
# # Books use case
# entity_matching_testsets= {
#     ("goodreads_small","amazon_small"): INPUT_DIR + "datasets/books/testsets/goodreads_2_amazon.csv",
#     ("metabooks_small","amazon_small"): INPUT_DIR + "datasets/books/testsets/metabooks_2_amazon.csv",
#     ("metabooks_small","goodreads_small"): INPUT_DIR + "datasets/books/testsets/metabooks_2_goodreads.csv",
# }

# datasets= [
#     INPUT_DIR + "datasets/books/amazon_small.parquet",
#     INPUT_DIR + "datasets/books/goodreads_small.parquet",
#     INPUT_DIR + "datasets/books/metabooks_small.parquet"
# ]
# fusion_testset= INPUT_DIR + "datasets/books/testsets/golden_fused_books.csv"

In [16]:
# Skip Blocking and Matching node and use existing strategies saved in 
# "output/blocking-evaluation/blocking_config.json" and "output/matching-evaluation/matching_config.json"
SKIP_BLOCKING_TESTER=True
SKIP_MATCHING_TESTER=True

In [17]:
# sys.setrecursionlimit(5000)
# print(sys.getrecursionlimit())

profile_tool = ProfileDatasetTool()
search_tool = SearchDocumentationTool()
all_tools = {profile_tool.name: profile_tool, search_tool.name: search_tool}

from workflow_logging import attach_logging
agent = SimpleModelAgent(llm, tools=all_tools)
attach_logging(agent, output_dir=OUTPUT_DIR, summary_model_name="gpt-4.1-mini", notebook_name="ClusterDoc")

# invoke agent
result = agent.graph.invoke({
    "datasets": datasets,
    "entity_matching_testsets": entity_matching_testsets,
    "fusion_testset": fusion_testset,
    "matcher_mode": "ml",
},
config={"recursion_limit": 10000})

[*] Running schema matching
[*] Schema alignment columns:
    discogs: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'label', 'genre', 'tracks_track_name', 'tracks_track_position', 'tracks_track_duration']
    lastfm: ['id', 'name', 'artist', 'duration', 'tracks_track_name', 'tracks_track_duration', 'tracks_track_position']
    musicbrainz: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'tracks_track_name', 'tracks_track_duration', 'tracks_track_position']
[*] Profiling datasets
TOKEN USAGE (profile_data):
   Prompt tokens: 287
   Completion tokens: 88
   Total tokens: 375
   Estimated cost: $0.000000
[+] Using saved blocking strategy: {
  "blocking_strategies": {
    "discogs_lastfm": {
      "strategy": "semantic_similarity",
      "columns": [
        "name",
        "artist"
      ],
      "params": {
        "top_k": 10
      },
      "pair_completeness": 0.9642857142857143,
      "num_candidates": 225831,
      "is_acceptable": t

/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_41309/1247590890.py:44: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)


[SEARCH TOOL CALL 2]: Asking 'SortedNeighbourhoodBlocker usage parameters key window id_column output_dir'
[SEARCH TOOL CALL 3]: Asking 'MLBasedMatcher.match parameters candidates trained_classifier id_column'
[SEARCH TOOL CALL 4]: Asking 'FeatureExtractor with StringComparator NumericComparator DateComparator usage'
[SEARCH TOOL CALL 5]: Asking 'DataFusionEngine.run parameters include_singletons id_column'
TOKEN USAGE (pipeline_adaption):
   Prompt tokens: 15,537
   Completion tokens: 3,299
   Total tokens: 18,836
   Estimated cost: $0.000000
[*] Running generated pipeline
[+] Pipeline successfully completed
[*] Running Cluster Analysis to get recommendations...
[*] Found 3 relevant correspondence files to analyze.


 CLUSTER ANALYSIS
[*] Analyzing clusters from correspondences_musicbrainz_lastfm.csv
    Total clusters: 4177
    Max cluster size: 2
    Cluster Distribution:
 cluster_size  frequency  percentage
            2       4177       100.0
    Small cluster ratio: 100.00%
    L

In [18]:
# from IPython.display import Image, display

# display(Image(agent.graph.get_graph().draw_mermaid_png()))